In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import datetime, timedelta
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend for saving images

class ChartDataCollector:
    def __init__(self, output_dir='dataset'):
        self.output_dir = output_dir
        self.create_folders()
        
    def create_folders(self):
        """Create folder structure for dataset"""
        folders = ['Uptrend', 'Downtrend', 'Sideways', 'raw_data']
        for folder in folders:
            os.makedirs(os.path.join(self.output_dir, folder), exist_ok=True)
    
    def download_stock_data(self, symbols, start_date='2018-01-01', end_date='2024-01-01'):
        """Download historical stock data"""
        all_data = {}
        
        for symbol in symbols:
            try:
                print(f"Downloading {symbol}...")
                data = yf.download(symbol, start=start_date, end=end_date, progress=False)
                if len(data) > 0:
                    all_data[symbol] = data
                    data.to_csv(f"{self.output_dir}/raw_data/{symbol}.csv")
                    print(f"  Downloaded {len(data)} records for {symbol}")
                else:
                    print(f"  No data for {symbol}")
            except Exception as e:
                print(f"Error downloading {symbol}: {e}")
        
        return all_data
    
    def create_chart_image(self, data, symbol, trend_type, 
                           chart_type='candlestick', save=True):
        """Create chart images from data"""
        
        fig, ax = plt.subplots(figsize=(10, 6), dpi=80)
        
        if chart_type == 'candlestick':
            # Simple OHLC representation
            dates = range(len(data))
            colors = ['green' if row['Close'] >= row['Open'] else 'red' 
                     for idx, row in data.iterrows()]
            
            for i, (idx, row) in enumerate(data.iterrows()):
                color = colors[i]
                
                # High-Low line
                ax.plot([i, i], [row['Low'], row['High']], 
                       color=color, linewidth=1)
                
                # Open-Close rectangle
                ax.plot([i-0.3, i+0.3], [row['Open'], row['Open']], 
                       color=color, linewidth=2)
                ax.plot([i-0.3, i+0.3], [row['Close'], row['Close']], 
                       color=color, linewidth=2)
                ax.fill_between([i-0.3, i+0.3], row['Open'], row['Close'], 
                               color=color, alpha=0.3)
        else:
            # Line chart
            ax.plot(data['Close'].values, linewidth=2, color='blue')
        
        # Customize chart
        ax.set_title(f'{symbol} - {trend_type}', fontsize=14)
        ax.set_ylabel('Price', fontsize=12)
        ax.grid(True, alpha=0.3)
        ax.set_xticks([])  # Hide x-axis for image classification
        ax.set_yticks([])  # Hide y-axis ticks
        
        if save:
            # Save image
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            filename = f"{symbol}_{trend_type}_{timestamp}.png"
            filepath = os.path.join(self.output_dir, trend_type, filename)
            plt.savefig(filepath, bbox_inches='tight', dpi=80, 
                       facecolor='white', edgecolor='none')
            plt.close(fig)
            return filepath
        else:
            return fig
    
    def calculate_trend_label(self, current_price, future_price, threshold=0.05):
        """Calculate trend label based on price movement"""
        if future_price is None or np.isnan(future_price):
            return None
            
        returns = (future_price - current_price) / current_price
        
        if returns > threshold:  # More than threshold increase
            return 'Uptrend'
        elif returns < -threshold:  # More than threshold decrease
            return 'Downtrend'
        else:  # Within threshold range
            return 'Sideways'
    
    def generate_labeled_dataset(self, data_dict, window_size=50, prediction_window=10):
        """Generate labeled dataset with sliding window"""
        
        images_info = []
        
        for symbol, df in data_dict.items():
            print(f"Processing {symbol}...")
            
            if len(df) < window_size + prediction_window:
                print(f"  Skipping {symbol}: insufficient data ({len(df)} records)")
                continue
            
            # Get price data
            prices = df['Close'].values
            
            # Slide window through data
            for i in range(window_size, len(prices) - prediction_window):
                # Extract window data
                start_idx = i - window_size
                end_idx = i
                
                window_df = df.iloc[start_idx:end_idx]
                current_price = prices[i]
                future_price = prices[i + prediction_window]
                
                # Determine trend based on future price
                trend = self.calculate_trend_label(current_price, future_price)
                
                if trend is None:
                    continue
                
                # Create chart image
                try:
                    img_path = self.create_chart_image(window_df, symbol, trend)
                    
                    images_info.append({
                        'symbol': symbol,
                        'start_date': window_df.index[0],
                        'end_date': window_df.index[-1],
                        'trend': trend,
                        'current_price': current_price,
                        'future_price': future_price,
                        'return': (future_price - current_price) / current_price,
                        'image_path': img_path
                    })
                    
                    # Print progress
                    if len(images_info) % 100 == 0:
                        print(f"  Generated {len(images_info)} images...")
                        
                except Exception as e:
                    print(f"  Error creating chart: {e}")
                    continue
        
        if len(images_info) > 0:
            # Save metadata
            metadata_df = pd.DataFrame(images_info)
            metadata_df.to_csv(f'{self.output_dir}/metadata.csv', index=False)
            print(f"\nGenerated {len(metadata_df)} images total")
            return metadata_df
        else:
            print("No images generated!")
            return pd.DataFrame()
    
    def collect_from_multiple_sources(self, num_symbols=5):
        """Collect data from various sources"""
        
        # List of symbols to collect (starting with a smaller set)
        symbols = [
            'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA',
            'JPM', 'BAC', 'GS', 'WMT', 'JNJ',
            'SPY', 'QQQ', 'DIA', 'VTI', 'VOO'
        ]
        
        # Use fewer symbols for testing
        symbols = symbols[:num_symbols]
        
        print(f"Downloading data for {len(symbols)} symbols...")
        data = self.download_stock_data(symbols)
        
        if not data:
            print("No data downloaded!")
            return pd.DataFrame()
        
        # Generate labeled dataset
        print("\nGenerating labeled chart images...")
        metadata = self.generate_labeled_dataset(data, window_size=30, prediction_window=5)
        
        if len(metadata) > 0:
            print(f"\nDataset created with {len(metadata)} images")
            print("\nTrend distribution:")
            print(metadata['trend'].value_counts())
            
            # Save dataset statistics
            stats = {
                'total_images': len(metadata),
                'uptrend_count': len(metadata[metadata['trend'] == 'Uptrend']),
                'downtrend_count': len(metadata[metadata['trend'] == 'Downtrend']),
                'sideways_count': len(metadata[metadata['trend'] == 'Sideways']),
                'unique_symbols': metadata['symbol'].nunique(),
                'date_range': f"{metadata['start_date'].min()} to {metadata['end_date'].max()}"
            }
            
            stats_df = pd.DataFrame([stats])
            stats_df.to_csv(f'{self.output_dir}/dataset_stats.csv', index=False)
            print(f"\nStatistics saved to {self.output_dir}/dataset_stats.csv")
        
        return metadata

# ============ SIMPLIFIED TEST VERSION ============
class SimpleChartDataCollector:
    """Simplified version for quick testing"""
    
    def __init__(self, output_dir='simple_dataset'):
        self.output_dir = output_dir
        self.create_folders()
    
    def create_folders(self):
        """Create folder structure"""
        for folder in ['Uptrend', 'Downtrend', 'Sideways']:
            os.makedirs(os.path.join(self.output_dir, folder), exist_ok=True)
    
    def generate_test_charts(self, num_per_class=50):
        """Generate synthetic test charts quickly"""
        import matplotlib.pyplot as plt
        import numpy as np
        
        images_info = []
        
        # Generate Uptrend charts
        print("Generating Uptrend charts...")
        for i in range(num_per_class):
            fig, ax = plt.subplots(figsize=(10, 6), dpi=80)
            
            # Create upward trend
            x = np.arange(50)
            y = 100 + x * 2 + np.random.randn(50) * 5
            
            ax.plot(y, color='green', linewidth=3)
            ax.fill_between(range(50), y-5, y+5, alpha=0.2, color='green')
            ax.set_title(f'Uptrend Pattern {i+1}')
            ax.grid(True, alpha=0.3)
            ax.set_xticks([])
            ax.set_yticks([])
            
            filename = f"uptrend_{i+1:03d}.png"
            filepath = os.path.join(self.output_dir, 'Uptrend', filename)
            plt.savefig(filepath, bbox_inches='tight')
            plt.close()
            
            images_info.append({
                'image_path': filepath,
                'trend': 'Uptrend',
                'type': 'synthetic'
            })
        
        # Generate Downtrend charts
        print("Generating Downtrend charts...")
        for i in range(num_per_class):
            fig, ax = plt.subplots(figsize=(10, 6), dpi=80)
            
            # Create downward trend
            x = np.arange(50)
            y = 200 - x * 2 + np.random.randn(50) * 5
            
            ax.plot(y, color='red', linewidth=3)
            ax.fill_between(range(50), y-5, y+5, alpha=0.2, color='red')
            ax.set_title(f'Downtrend Pattern {i+1}')
            ax.grid(True, alpha=0.3)
            ax.set_xticks([])
            ax.set_yticks([])
            
            filename = f"downtrend_{i+1:03d}.png"
            filepath = os.path.join(self.output_dir, 'Downtrend', filename)
            plt.savefig(filepath, bbox_inches='tight')
            plt.close()
            
            images_info.append({
                'image_path': filepath,
                'trend': 'Downtrend',
                'type': 'synthetic'
            })
        
        # Generate Sideways charts
        print("Generating Sideways charts...")
        for i in range(num_per_class):
            fig, ax = plt.subplots(figsize=(10, 6), dpi=80)
            
            # Create sideways trend
            x = np.arange(50)
            y = 150 + np.sin(x * 0.3) * 10 + np.random.randn(50) * 3
            
            ax.plot(y, color='blue', linewidth=3)
            ax.fill_between(range(50), y-5, y+5, alpha=0.2, color='blue')
            ax.set_title(f'Sideways Pattern {i+1}')
            ax.grid(True, alpha=0.3)
            ax.set_xticks([])
            ax.set_yticks([])
            
            filename = f"sideways_{i+1:03d}.png"
            filepath = os.path.join(self.output_dir, 'Sideways', filename)
            plt.savefig(filepath, bbox_inches='tight')
            plt.close()
            
            images_info.append({
                'image_path': filepath,
                'trend': 'Sideways',
                'type': 'synthetic'
            })
        
        # Save metadata
        metadata_df = pd.DataFrame(images_info)
        metadata_df.to_csv(f'{self.output_dir}/metadata.csv', index=False)
        
        print(f"\nGenerated {len(metadata_df)} synthetic images")
        print("Distribution:")
        print(metadata_df['trend'].value_counts())
        
        return metadata_df

# ============ QUICK TEST EXECUTION ============
if __name__ == "__main__":
    print("=" * 60)
    print("TREND IDENTIFICATION DATASET COLLECTOR")
    print("=" * 60)
    
    print("\nChoose collection method:")
    print("1. Quick test with synthetic data (fast)")
    print("2. Real stock data with yfinance (slower, needs internet)")
    
    choice = input("\nEnter choice (1 or 2): ").strip()
    
    if choice == '1':
        print("\nGenerating synthetic dataset...")
        collector = SimpleChartDataCollector(output_dir='synthetic_dataset')
        metadata = collector.generate_test_charts(num_per_class=50)
        print(f"\nDataset saved to: synthetic_dataset/")
        
    elif choice == '2':
        print("\nCollecting real stock data...")
        try:
            # Test with just 2 symbols first
            collector = ChartDataCollector(output_dir='real_stock_dataset')
            metadata = collector.collect_from_multiple_sources(num_symbols=2)
            
            if len(metadata) > 0:
                print(f"\nDataset saved to: real_stock_dataset/")
            else:
                print("\nNo data collected. Trying synthetic instead...")
                collector = SimpleChartDataCollector(output_dir='fallback_dataset')
                metadata = collector.generate_test_charts(num_per_class=30)
                
        except Exception as e:
            print(f"\nError collecting real data: {e}")
            print("Falling back to synthetic data...")
            collector = SimpleChartDataCollector(output_dir='fallback_dataset')
            metadata = collector.generate_test_charts(num_per_class=30)
    
    else:
        print("\nInvalid choice. Using synthetic data...")
        collector = SimpleChartDataCollector(output_dir='default_dataset')
        metadata = collector.generate_test_charts(num_per_class=30)
    
    print("\n" + "=" * 60)
    print("DATASET READY FOR TRAINING!")
    print("=" * 60)
    print("\nNext steps:")
    print("1. Use the generated images in your training pipeline")
    print("2. Check metadata.csv for image paths and labels")
    print("3. You can now run the TrendIdentificationPipeline")

TREND IDENTIFICATION DATASET COLLECTOR

Choose collection method:
1. Quick test with synthetic data (fast)
2. Real stock data with yfinance (slower, needs internet)



Enter choice (1 or 2):  1



Generating synthetic dataset...
Generating Uptrend charts...
Generating Downtrend charts...
Generating Sideways charts...

Generated 150 synthetic images
Distribution:
trend
Uptrend      50
Downtrend    50
Sideways     50
Name: count, dtype: int64

Dataset saved to: synthetic_dataset/

DATASET READY FOR TRAINING!

Next steps:
1. Use the generated images in your training pipeline
2. Check metadata.csv for image paths and labels
3. You can now run the TrendIdentificationPipeline
